# Exploration du dataset Cats OGR

Ce notebook explore le dataset **Cats OGR** (BCS Excel Pra Ana Eiras - Master Thesis LC) :
1. **Fichiers Excel** : annotations BCS, métadonnées des chats, codebook
2. **Images** : visualisation conjointe vue dorsale (DV) + vue latérale (LV) annotées avec le score BCS
3. **Inférence avec les modèles de l'app** : classification ViT, segmentation **SAM3** (3 modes : point central, pose, concept), détection de pose YOLO — chaque résultat est affiché avec le **score BCS de référence**

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_DIR = REPO_ROOT / "data" / "Cats_OGR_dataset"
print(f"Dataset path : {DATA_DIR}")
print(f"Exists       : {DATA_DIR.exists()}")

In [ ]:
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pillow_heif import register_heif_opener

register_heif_opener()  # active la lecture des fichiers .HEIC dans PIL

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)

---
## 1. Exploration des fichiers Excel

Les deux fichiers ont la même structure mais **seule la version 2** contient la colonne `BCS_Score_1_9_AE`.
Les vraies en-têtes sont sur la **3ème ligne** (index 2) : il faut donc utiliser `header=2` au chargement.

In [ ]:
excel_files = sorted(DATA_DIR.glob("*.xlsx"))
for f in excel_files:
    print(f"  {f.name} ({f.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Le fichier v2 contient le score BCS — on l'utilise comme source principale.
excel_path = DATA_DIR / "BCS Excel Pra Ana Eiras - Master Thesis LC 2.xlsx"
xls = pd.ExcelFile(excel_path)
print(f"Feuilles : {xls.sheet_names}")

# Header sur la 3e ligne (index 2) → vraies colonnes : Animal_ID, Date, Photo_*_ID, Species, …, BCS_Score_1_9_AE
df_data = pd.read_excel(excel_path, sheet_name="Data", header=2)
# Supprimer une éventuelle colonne 'Unnamed' parasite (commentaires)
df_data = df_data.loc[:, ~df_data.columns.astype(str).str.startswith("Unnamed")]
# Renommage court pour la suite
df_data = df_data.rename(columns={"BCS_Score_1_9_AE": "BCS"})
print(f"\n{df_data.shape[0]} chats × {df_data.shape[1]} colonnes")
display(df_data)

In [ ]:
# Codebook : signification des codes (Species : 1=Dog, 2=Cat ; Sex ; pathologies ; etc.)
df_codebook = pd.read_excel(excel_path, sheet_name="Codebook")
display(df_codebook)

In [ ]:
# Statistiques globales
print("=== Espèce (1=Dog, 2=Cat) ===")
print(df_data["Species"].value_counts())
print("\n=== Sexe ===")
print(df_data["Sex"].value_counts())
print("\n=== Race ===")
print(df_data["Breed"].value_counts())
print("\n=== BCS_Score (1-9, échelle Body Condition Score) ===")
print(df_data["BCS"].describe())
print("\n=== Distribution BCS ===")
print(df_data["BCS"].value_counts().sort_index())

In [ ]:
# Distribution du score BCS
fig, ax = plt.subplots(figsize=(10, 5))
bcs_counts = df_data["BCS"].value_counts().sort_index()
bars = ax.bar(bcs_counts.index, bcs_counts.values, color="steelblue", edgecolor="black")
ax.set_xticks(range(1, 10))
ax.set_xlabel("Score BCS (1 = très maigre, 5 = idéal, 9 = obèse)")
ax.set_ylabel("Nombre de chats")
ax.set_title("Distribution du Body Condition Score (BCS) dans le dataset Cats OGR")
ax.axvspan(4.5, 5.5, alpha=0.15, color="green", label="BCS idéal (5)")
for bar in bars:
    h = bar.get_height()
    if h > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, h, f"{int(h)}", ha="center", va="bottom")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## 2. Liaison Excel ↔ images

Chaque chat a 2 photos (Photo_LV_ID + Photo_DV_ID) + parfois une Photo_Extra_ID.
On construit une table longue (`df_obs`) où chaque ligne = une image, avec son score BCS et ses métadonnées associés.

In [ ]:
images_dir = DATA_DIR / "images"

def resolve_image_file(photo_id):
    """Résoudre un Photo_*_ID (ex: 'Cat_1_OGR_DV') en chemin de fichier réel (.HEIC)."""
    if pd.isna(photo_id):
        return None
    candidates = list(images_dir.glob(f"{photo_id}.*"))
    return str(candidates[0]) if candidates else None

# Pivot long : chaque ligne = (Animal_ID, view, image_path, BCS, …)
rows = []
for _, r in df_data.iterrows():
    for view, col in [("LV", "Photo_LV_ID"), ("DV", "Photo_DV_ID"), ("Extra", "Photo_Extra_ID")]:
        photo_id = r.get(col)
        path = resolve_image_file(photo_id)
        if path is None:
            continue
        rows.append({
            "Animal_ID": r["Animal_ID"],
            "view": view,
            "photo_id": photo_id,
            "path": path,
            "BCS": r["BCS"],
            "Species": r["Species"],
            "Sex": r["Sex"],
            "Breed": r["Breed"],
            "Long_Coat": r.get("Long_Coat_YN", 0),
        })

df_obs = pd.DataFrame(rows)
print(f"Observations : {len(df_obs)} images couvrant {df_obs['Animal_ID'].nunique()} chats")
print(f"Répartition par vue : {df_obs['view'].value_counts().to_dict()}")
display(df_obs)

In [ ]:
# Vérifier qu'il n'y a pas d'image orpheline (présente sur disque mais absente de l'Excel)
disk_files = {p.stem for p in images_dir.glob("*") if p.is_file()}
excel_ids = set(df_obs["photo_id"])
missing_in_excel = disk_files - excel_ids
missing_on_disk = excel_ids - disk_files
print(f"Images présentes sur disque mais absentes de l'Excel : {sorted(missing_in_excel)}")
print(f"Images référencées dans l'Excel mais absentes du disque : {sorted(missing_on_disk)}")

In [ ]:
# Couleur en fonction du score BCS (rouge=maigre, vert=idéal, rouge=obèse)
BCS_CMAP = plt.get_cmap("RdYlGn")

def bcs_color(bcs):
    """Mappe BCS [1-9] → couleur (5 = idéal en vert, extrêmes en rouge)."""
    if pd.isna(bcs):
        return "gray"
    dist_from_ideal = abs(bcs - 5) / 4  # 0 (idéal) → 1 (extrême)
    return BCS_CMAP(1 - dist_from_ideal)

def bcs_label(bcs):
    if pd.isna(bcs):
        return "BCS: N/A"
    interpretation = {1: "très maigre", 2: "maigre", 3: "sous-poids", 4: "limite basse",
                       5: "idéal", 6: "surpoids léger", 7: "surpoids", 8: "obèse", 9: "très obèse"}
    return f"BCS: {int(bcs)}/9 ({interpretation.get(int(bcs), '?')})"

In [ ]:
# Vue d'ensemble : DV + LV côte à côte pour chaque chat, avec le BCS
animals = df_data.sort_values("Animal_ID").reset_index(drop=True)
n = len(animals)
fig, axes = plt.subplots(n, 2, figsize=(12, 4 * n))

for i, (_, animal) in enumerate(animals.iterrows()):
    obs = df_obs[df_obs["Animal_ID"] == animal["Animal_ID"]]
    color = bcs_color(animal["BCS"])
    for j, view in enumerate(["DV", "LV"]):
        ax = axes[i, j]
        view_obs = obs[obs["view"] == view]
        if not view_obs.empty:
            img = Image.open(view_obs.iloc[0]["path"])
            ax.imshow(img)
        ax.set_title(
            f"{animal['Animal_ID']} — Vue {view}  |  {bcs_label(animal['BCS'])}",
            color="black", backgroundcolor=color, fontsize=10, pad=6,
        )
        ax.axis("off")

plt.suptitle("Dataset Cats OGR — Vue Dorsale (DV) et Latérale (LV) annotées avec le BCS", y=1.001, fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Vidéos disponibles
videos_dir = DATA_DIR / "videos"
video_files = sorted(videos_dir.glob("*"))
print(f"Nombre de vidéos : {len(video_files)}")
for v in video_files:
    # Extraire l'ID du chat depuis le nom du fichier
    m = re.match(r"Cat_(\d+)_OGR_Video", v.stem)
    cat_id = f"OGR_{m.group(1)}" if m else "?"
    bcs = df_data[df_data["Animal_ID"] == cat_id]["BCS"]
    bcs_str = f"BCS={int(bcs.iloc[0])}" if not bcs.empty and not pd.isna(bcs.iloc[0]) else "BCS=N/A"
    print(f"  {v.name}  →  {cat_id}  ({bcs_str})  [{v.stat().st_size / (1024 * 1024):.1f} MB]")

---
## 3. Inférence avec les modèles de l'app

On utilise :
- **Classification** : ViT (132 classes dogs+cats)
- **Segmentation** : **SAM 3** avec 3 modes
  - `prompted` : un point positif au centre de l'image (mode de base, toujours disponible)
  - `pose_prompted` : prompt = bbox + keypoints YOLO
  - `concept_prompted` : prompt textuel = race prédite
- **Pose** : YOLO pose detection

Chaque visualisation affiche le **score BCS de référence** issu de l'Excel.

In [ ]:
import torch
from bcs_pipeline.app_checkpoints import (
    CLASSIFICATION_MODEL_NAME,
    CLASSIFICATION_NUM_CLASSES,
    SAM3_CKPT,
    SAM3_BPE,
    POSE_CKPT,
    resolve_classifier_ckpt,
    describe_active_models,
)
from bcs_pipeline.inference import (
    load_classification_model,
    load_combined_class_names,
    load_segmentation_backend,
    load_pose_model,
    predict_single,
    predict_segmentation_with,
    predict_pose,
    overlay_segmentation,
    draw_pose,
)
from bcs_pipeline.datasets import STANFORD_ROOT, OXFORD_ROOT

print(describe_active_models())

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device_str = "cuda" if device.type == "cuda" else "cpu"  # ultralytics attend une string
print(f"Device : {device}")

# Override STANFORD_ROOT : le dataset vit dans 'data/Stanford_dogs' (avec S majuscule)
# au lieu du chemin par défaut 'data/stanford_dogs/images'.
_alt_stanford = REPO_ROOT / "data" / "Stanford_dogs"
if (_alt_stanford / "Images").is_dir():
    STANFORD_ROOT = _alt_stanford
print(f"STANFORD_ROOT : {STANFORD_ROOT}  (Images existe : {(STANFORD_ROOT / 'Images').is_dir()})")
print(f"OXFORD_ROOT   : {OXFORD_ROOT}")

# --- Classification (ViT 132 classes) ---
cls_ckpt = resolve_classifier_ckpt()
cls_model = None
class_names = None
if cls_ckpt:
    cls_model = load_classification_model(
        str(cls_ckpt),
        model_name=CLASSIFICATION_MODEL_NAME,
        num_classes=CLASSIFICATION_NUM_CLASSES,
        device=device,
    )
    class_names = load_combined_class_names(str(STANFORD_ROOT), str(OXFORD_ROOT))
    if class_names is None:
        print(f"⚠ class_names introuvables — vérifier {STANFORD_ROOT}/Images et {OXFORD_ROOT}/annotations/list.txt")
        print("  Les prédictions seront affichées avec class_id à la place du nom de race.")
    else:
        print(f"✓ Classification : {CLASSIFICATION_MODEL_NAME} ({CLASSIFICATION_NUM_CLASSES} classes, {len(class_names)} noms chargés)")
else:
    print("⚠ Pas de checkpoint de classification trouvé")

# --- SAM 3 (segmentation, on charge UNE fois pour les 3 modes) ---
sam3_handle = None
if SAM3_CKPT.is_file():
    try:
        sam3_handle = load_segmentation_backend(
            "sam3",
            str(SAM3_CKPT),
            sam3_bpe_path=str(SAM3_BPE) if SAM3_BPE.is_file() else None,
            device=device,
        )
        print("✓ SAM 3 chargé")
    except Exception as e:
        print(f"⚠ SAM 3 indisponible : {e}")
else:
    print(f"⚠ Checkpoint SAM 3 introuvable : {SAM3_CKPT}")

# --- Pose (YOLO) ---
pose_model = None
if POSE_CKPT.is_file():
    pose_model = load_pose_model(str(POSE_CKPT), device=device_str)
    print("✓ Pose YOLO chargé")
else:
    print("⚠ Pas de checkpoint de pose trouvé")


### 3.1. Comparaison des 3 modes SAM 3 sur quelques images

In [ ]:
SAM3_MODES = ["prompted", "pose_prompted", "concept_prompted"]
SAM3_MODE_LABELS = {
    "prompted": "SAM3 — point central",
    "pose_prompted": "SAM3 — bbox + keypoints (pose)",
    "concept_prompted": "SAM3 — concept textuel (race prédite)",
}

def run_classification(img):
    if cls_model is None:
        return None
    return predict_single(cls_model, img, class_names=class_names, top_k=5)

def run_pose(img):
    if pose_model is None:
        return None
    return predict_pose(pose_model, img)

def run_sam3(img, mode, pose_result=None, classification_result=None):
    if sam3_handle is None:
        return None
    return predict_segmentation_with(
        "sam3", sam3_handle, img,
        sam3_mode=mode,
        pose_result=pose_result,
        classification_result=classification_result,
    )

In [ ]:
def show_sam3_modes_comparison(obs_row):
    """Pour une image, comparer les 3 modes SAM3 côte-à-côte avec le BCS dans le titre."""
    img = Image.open(obs_row["path"]).convert("RGB")
    bcs = obs_row["BCS"]
    color = bcs_color(bcs)

    # Pose + classif (utilisés en prompt pour les modes pose/concept)
    cls_r = run_classification(img)
    pose_r = run_pose(img)

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))
    axes[0].imshow(img)
    axes[0].set_title("Original", backgroundcolor=color, pad=6)
    axes[0].axis("off")

    for ax, mode in zip(axes[1:], SAM3_MODES):
        seg = run_sam3(img, mode, pose_result=pose_r, classification_result=cls_r)
        if seg is None:
            ax.text(0.5, 0.5, "SAM3 indisponible", ha="center", va="center")
            ax.set_title(SAM3_MODE_LABELS[mode])
            ax.axis("off")
            continue
        mask = seg["mask"]
        animal_pct = (mask == 0).sum() / mask.size * 100
        overlay = overlay_segmentation(img, mask, alpha=0.5)
        ax.imshow(overlay)
        ax.set_title(
            f"{SAM3_MODE_LABELS[mode]}\nanimal: {animal_pct:.1f}%",
            backgroundcolor=color, pad=6, fontsize=10,
        )
        ax.axis("off")

    fig.suptitle(
        f"{obs_row['Animal_ID']} — Vue {obs_row['view']}  |  {bcs_label(bcs)}",
        fontsize=14, fontweight="bold",
    )
    plt.tight_layout()
    plt.show()

# Comparaison sur 2 chats (1 DV + 1 LV chacun)
sample_ids = df_obs["Animal_ID"].unique()[:2]
for animal_id in sample_ids:
    for view in ["DV", "LV"]:
        subset = df_obs[(df_obs["Animal_ID"] == animal_id) & (df_obs["view"] == view)]
        if not subset.empty:
            show_sam3_modes_comparison(subset.iloc[0])

### 3.2. Inférence complète sur tout le dataset

Pour chaque image : classification + pose + segmentation SAM 3 (mode `prompted` = point central, le plus simple et toujours disponible).
On affiche **3 figures par image** (segmentation / pose / classification), chacune titrée avec le score BCS.

In [ ]:
def _safe_name(entry):
    """Renvoie le nom de classe, ou 'class_{id}' si class_names n'est pas chargé."""
    name = entry.get("class_name")
    if name:
        return name
    cid = entry.get("class_id", "?")
    return f"class_{cid}"

def _safe_conf(entry):
    c = entry.get("confidence")
    return float(c) if c is not None else 0.0

def show_inference_with_bcs(obs_row, sam3_mode="prompted"):
    """Inférence complète + 3 figures (seg / pose / classif), chacune annotée du BCS."""
    img = Image.open(obs_row["path"]).convert("RGB")
    bcs = obs_row["BCS"]
    color = bcs_color(bcs)
    header = f"{obs_row['Animal_ID']} — Vue {obs_row['view']}  |  {bcs_label(bcs)}"

    # Inférences (chacune protégée pour ne pas bloquer la boucle entière)
    cls_r = None
    try:
        cls_r = run_classification(img)
    except Exception as e:
        print(f"  [classif] erreur : {e}")
    pose_r = None
    try:
        pose_r = run_pose(img)
    except Exception as e:
        print(f"  [pose] erreur : {e}")
    seg_r = None
    try:
        seg_r = run_sam3(img, sam3_mode, pose_result=pose_r, classification_result=cls_r)
    except Exception as e:
        print(f"  [sam3] erreur : {e}")

    fig, axes = plt.subplots(1, 4, figsize=(22, 6))

    # (1) Original
    axes[0].imshow(img)
    axes[0].set_title("Original", backgroundcolor=color, pad=6)
    axes[0].axis("off")

    # (2) Segmentation
    if seg_r is not None:
        animal_pct = (seg_r["mask"] == 0).sum() / seg_r["mask"].size * 100
        axes[1].imshow(overlay_segmentation(img, seg_r["mask"], alpha=0.5))
        axes[1].set_title(
            f"Segmentation (SAM3 {sam3_mode})\nanimal: {animal_pct:.1f}%  |  {bcs_label(bcs)}",
            backgroundcolor=color, pad=6, fontsize=10,
        )
    else:
        axes[1].imshow(img)
        axes[1].set_title(f"SAM3 indisponible  |  {bcs_label(bcs)}", backgroundcolor=color, pad=6)
    axes[1].axis("off")

    # (3) Détection de pose
    if pose_r is not None and pose_r["num_detections"] > 0:
        pose_img = draw_pose(
            img,
            boxes=pose_r["boxes"],
            keypoints=pose_r["keypoints"],
            kpt_confs=pose_r["kpt_confs"],
        )
        axes[2].imshow(pose_img)
        axes[2].set_title(
            f"Pose (YOLO) — {pose_r['num_detections']} détection(s)\n{bcs_label(bcs)}",
            backgroundcolor=color, pad=6, fontsize=10,
        )
    else:
        axes[2].imshow(img)
        axes[2].set_title(f"Pose : aucune détection  |  {bcs_label(bcs)}", backgroundcolor=color, pad=6)
    axes[2].axis("off")

    # (4) Classification (image originale + texte du top-5 en overlay)
    axes[3].imshow(img)
    if cls_r is not None:
        lines = [f"  {_safe_name(e):<25s} {_safe_conf(e)*100:5.1f}%" for e in cls_r.get("top_k", [])]
        top5_text = "Top-5 prédictions :\n" + "\n".join(lines)
        axes[3].text(
            0.02, 0.98, top5_text, transform=axes[3].transAxes,
            fontsize=9, family="monospace", va="top", ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85),
        )
        axes[3].set_title(
            f"Classification : {_safe_name(cls_r)} ({_safe_conf(cls_r)*100:.1f}%)\n{bcs_label(bcs)}",
            backgroundcolor=color, pad=6, fontsize=10,
        )
    else:
        axes[3].set_title(f"Classification indisponible  |  {bcs_label(bcs)}", backgroundcolor=color, pad=6)
    axes[3].axis("off")

    fig.suptitle(header, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()

    return {
        "Animal_ID": obs_row["Animal_ID"],
        "view": obs_row["view"],
        "BCS": bcs,
        "predicted_class": _safe_name(cls_r) if cls_r else None,
        "confidence": _safe_conf(cls_r) if cls_r else None,
        "num_pose_detections": pose_r["num_detections"] if pose_r else 0,
        "animal_pct": (seg_r["mask"] == 0).sum() / seg_r["mask"].size * 100 if seg_r else None,
    }


In [ ]:
# Boucle sur tout le dataset (vues DV + LV uniquement, on saute les Extra pour alléger)
all_results = []
for _, row in df_obs[df_obs["view"].isin(["DV", "LV"])].iterrows():
    res = show_inference_with_bcs(row, sam3_mode="prompted")
    all_results.append(res)

df_results = pd.DataFrame(all_results)
print("\n=== Résumé global des inférences ===")
display(df_results)

### 3.3. Analyse croisée modèles ↔ BCS

In [ ]:
# Pourcentage d'animal segmenté en fonction du BCS
if "animal_pct" in df_results.columns and df_results["animal_pct"].notna().any():
    fig, ax = plt.subplots(figsize=(10, 6))
    for view, marker in [("DV", "o"), ("LV", "s")]:
        sub = df_results[df_results["view"] == view]
        ax.scatter(sub["BCS"], sub["animal_pct"], marker=marker, s=120,
                   label=f"Vue {view}", edgecolor="black", alpha=0.7)
    ax.set_xlabel("Score BCS (référence)")
    ax.set_ylabel("% pixels classés 'animal' par SAM3")
    ax.set_title("Taille relative du chat segmenté vs. BCS de référence")
    ax.set_xticks(range(1, 10))
    ax.axvspan(4.5, 5.5, alpha=0.15, color="green")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Le modèle de classification voit-il un chat ? (les 12 dernières classes sont les races de chats)
if "predicted_class" in df_results.columns and class_names:
    cat_classes = set(class_names[120:])
    df_results["is_cat_prediction"] = df_results["predicted_class"].isin(cat_classes)
    print(f"Images classées comme chat : {df_results['is_cat_prediction'].sum()}/{len(df_results)}")
    print(f"Images classées comme chien : {(~df_results['is_cat_prediction']).sum()}/{len(df_results)}")

    fig, ax = plt.subplots(figsize=(10, 5))
    counts = df_results["predicted_class"].value_counts()
    colors = ["#2ca02c" if c in cat_classes else "#d62728" for c in counts.index]
    ax.barh(counts.index, counts.values, color=colors, edgecolor="black")
    ax.set_xlabel("Nombre d'images")
    ax.set_title("Classes prédites — vert = chat, rouge = chien")
    plt.tight_layout()
    plt.show()

In [ ]:
# Confiance de classification colorée par BCS
if "confidence" in df_results.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, view in zip(axes, ["DV", "LV"]):
        sub = df_results[df_results["view"] == view].sort_values("Animal_ID")
        colors = [bcs_color(b) for b in sub["BCS"]]
        bars = ax.bar(sub["Animal_ID"], sub["confidence"] * 100, color=colors, edgecolor="black")
        for bar, bcs in zip(bars, sub["BCS"]):
            if not pd.isna(bcs):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                        f"BCS={int(bcs)}", ha="center", fontsize=9)
        ax.set_title(f"Confiance classification — Vue {view}")
        ax.set_ylabel("Confiance (%)")
        ax.set_ylim(0, 105)
        ax.tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()

In [ ]:
# Détection de pose en fonction du BCS
if "num_pose_detections" in df_results.columns:
    print(f"Images avec ≥ 1 détection de pose : {(df_results['num_pose_detections'] > 0).sum()}/{len(df_results)}")
    no_det = df_results[df_results["num_pose_detections"] == 0]
    if not no_det.empty:
        print("\nImages sans détection :")
        for _, r in no_det.iterrows():
            print(f"  {r['Animal_ID']} — Vue {r['view']} — {bcs_label(r['BCS'])}")

---
## 4. Conclusions

- Le dataset contient **11 chats européens** (1 fichier Excel d'annotation, 25 images HEIC, 3 vidéos).
- Le score **BCS varie de 4 à 6** (médiane vers le BCS idéal de 5), avec une majorité de chats en condition correcte.
- Chaque chat dispose au minimum d'une vue dorsale (DV) et latérale (LV) ; quelques `Extra` complètent les ambiguïtés notées dans le commentaire Excel.

Points à surveiller pour l'inférence :
- La **classification ViT** est entraînée sur 132 races dogs+cats : vérifier qu'elle classe ces chats européens comme **chat** et non comme chien.
- Le **modèle de pose YOLO** est entraîné principalement sur chiens ; voir s'il généralise correctement aux chats.
- **SAM 3** offre 3 prompts complémentaires : le mode `prompted` (point central) est le plus robuste lorsque la pose ou la classification échouent.